In [1]:
%run 0_1_load_paths.ipynb

In [2]:
import os
import subprocess

import credentials
import momapy_kb.lpg.backends.neo4j
import momapy_kb.lpg.session

In [3]:
backend = momapy_kb.lpg.backends.neo4j.Neo4jBackend(
    hostname=credentials.NEO4J_URI,
    username=credentials.NEO4J_USERNAME,
    password=credentials.NEO4J_PASSWORD,
    notifications_min_severity="off",
)
session = momapy_kb.lpg.session.Session(backend)

<div class="alert alert-danger">We delete all the data from the DB</div>

In [4]:
session.delete_all()

## Adding the BEL KGs (AD, PD, COVID) to the Neo4j database

We define a function that imports a BEL KG cypher dump and wires it to a `Collection`/`CollectionEntry`/`BELModel`, then call it once per KG. Every step is scoped to the KG being imported (via a temporary `_NewImport` label and the collection name), so the three KGs are not cross-linked.

In [5]:
def save_bel_kg_from_file_path(session, collection_name, input_file_path):
    """Import a BEL KG cypher dump and wire it to a Collection/BELModel.

    Safe to call several times against the same database: every step is
    scoped to the nodes of the KG being imported (via a temporary
    `_NewImport` label and the collection name), so distinct KGs are not
    cross-linked.
    """
    # We import the cypher dump:
    command = [
        "cat",
        str(input_file_path),
        "|",
        "cypher-shell",
        "-a",
        credentials.NEO4J_URI,
        "-u",
        credentials.NEO4J_USERNAME,
        "-p",
        credentials.NEO4J_PASSWORD,
        "-d",
        credentials.NEO4J_DATABASE,
    ]
    result = subprocess.run(" ".join(command), shell=True)
    result.check_returncode()
    # We label the freshly imported nodes as BELModelElement and tag them with
    # a temporary _NewImport label to scope the following steps to this KG only
    # (the NOT n:BELModelElement guard leaves previously imported KGs alone):
    query = """
        MATCH (n)
        WHERE NOT n:Collection AND NOT n:CollectionEntry AND NOT n:BELModel
            AND NOT n:BELModelElement
        SET n:BELModelElement, n:_NewImport
        RETURN n
    """
    _ = session.execute_query(query)
    # We make the Collection, CollectionEntry and BELModel nodes:
    query = f"""
        MERGE
            (collection:Collection {{name: '{collection_name}'}})-[:HAS_ENTRY]->(collection_entry:CollectionEntry {{file_path: '{input_file_path}'}})-[:HAS_OBJ]->(model:BELModel)
        RETURN
            collection, collection_entry, model
    """
    _ = session.execute_query(query)
    # We link each newly imported node to this KG's model:
    query = f"""
        MATCH (:Collection {{name: '{collection_name}'}})-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(model:BELModel)
        MATCH (model_element:_NewImport)
        MERGE (model)-[:HAS_NODE]->(model_element)
        RETURN model, model_element
    """
    _ = session.execute_query(query)
    # We extract subgraph information from this KG's relationships and make
    # Subgraph nodes attached to its model:
    query = f"""
        MATCH (:Collection {{name: '{collection_name}'}})-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(model:BELModel)
        MATCH (n:_NewImport)-[r]->(m)
        UNWIND r.annotationSubgraph AS subgraph
        MERGE (model)-[:HAS_SUBGRAPH]->(subgraph_node:Subgraph {{name: subgraph}})
        RETURN subgraph_node
    """
    _ = session.execute_query(query)
    # We add this KG's nodes to its subgraphs:
    query = f"""
    MATCH (:Collection {{name: '{collection_name}'}})-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(model:BELModel)
    CALL (model) {{
        MATCH (model)-[:HAS_NODE]->(n)-[r]->(m)
        UNWIND r.annotationSubgraph AS subgraph
        RETURN n AS n, subgraph AS subgraph
        UNION
        MATCH (model)-[:HAS_NODE]->(n)<-[r]-(m)
        UNWIND r.annotationSubgraph AS subgraph
        RETURN n AS n, subgraph AS subgraph
    }}
    MATCH (model)-[:HAS_SUBGRAPH]->(subgraph_node:Subgraph)
    WHERE subgraph_node.name = subgraph
    MERGE (subgraph_node)-[:HAS_NODE]->(n)
    RETURN subgraph_node, n
    """
    _ = session.execute_query(query)
    # We make a special "main_model" Subgraph node for this model (so that
    # later queries are uniform for nodes which do not belong to a subgraph):
    query = f"""
        MATCH (:Collection {{name: '{collection_name}'}})-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(model:BELModel)
        MERGE (model)-[:HAS_SUBGRAPH]->(subgraph_node:Subgraph {{name: 'main_model'}})
        RETURN subgraph_node
    """
    _ = session.execute_query(query)
    # We add all of this KG's nodes that do not belong to a subgraph to the
    # "main_model" Subgraph node:
    query = f"""
        MATCH (:Collection {{name: '{collection_name}'}})-[:HAS_ENTRY]->(:CollectionEntry)-[:HAS_OBJ]->(model:BELModel)-[:HAS_SUBGRAPH]->(subgraph_node:Subgraph {{name: 'main_model'}})
        MATCH (model)-[:HAS_NODE]->(n:_NewImport)
        WHERE NOT EXISTS {{(n)<-[:HAS_NODE]-(s:Subgraph)}}
        MERGE (subgraph_node)-[:HAS_NODE]->(n)
        RETURN n
    """
    _ = session.execute_query(query)
    # We remove the temporary _NewImport label:
    query = """
        MATCH (n:_NewImport)
        REMOVE n:_NewImport
        RETURN n
    """
    _ = session.execute_query(query)

In [6]:
save_bel_kg_from_file_path(session, "AD_KG_BEL", AD_KG_CYPHER_DATA_FILE)
save_bel_kg_from_file_path(session, "PD_KG_BEL", PD_KG_CYPHER_DATA_FILE)
save_bel_kg_from_file_path(session, "COVID_KG_BEL", COVID_KG_CYPHER_DATA_FILE)

## Adding the COVID-19 DM and PD DM to the Neo4j database

We save the collection to the DB:

In [7]:
collection_names_and_input_file_paths = [
    (
        "COVID_DM_CD",
        COVID_DM_CD_DATA_DIR.glob("*.xml"),
    ),
    (
        "PD_DM_CD",
        PD_DM_CD_DATA_DIR.glob("*.xml"),
    ),
]

In [8]:
session.save_collections_from_file_paths(
    collection_names_and_input_file_paths,
    return_type="map",
    with_membership_edges=True,
    integration_mode="hash",
)

/home/rougny/code/momapy/src/momapy/celldesigner/io/celldesigner/reader.py:740: UserWarning: skipping modulation 'ir3e6': references a Degraded species (source alias='csa163', target alias='ir15b'); Degraded species have no model peer.
  cls._make_and_add_modulation(
